# 8.16 範圍、格式與結束，如何各自教與檢查？


書店的告示仍是四行：第1列「今日休館」、第2列「Closed today」、第3列「明日開放」、第4列「Open tomorrow」。現在收到的完整要求是：「只抄兩行中文，依原順序，回一個只有text欄位的JSON物件，欄位值是文字清單；JSON外不要加字。」

一份合格的可見回答如下，之後還要正常結束：

```json
{"text":["今日休館","明日開放"]}
```

先看文字內容：兩句應抄的中文有沒有錯字、漏字或交換順序？再看範圍：有沒有把英文或其他說明也放進回答？格式另問整份文字是否就是JSON物件、是否只有text欄位、值是否為字串清單。這裡的格式規則不靠「清單一定兩項」代替範圍檢查；多抄四行仍可能是格式正確的清單。

最後核對正常結束。它不是檢查最後一個可見字是否為右大括號。需要查看模型實際生成的結束token與停止原因，確認回答之後由模型選出EOS，而非上限用完或程式強制停下。若EOS提早出現，內容可能尚未完成；若文字都已寫完但沒有EOS，這次也不能記成主動結束。

圖中的四張檢查卡都指向同一題。每張卡只回答自己的問題，最下面要求四項同時通過；它畫的是驗收關係，不是在宣稱模型內有四個獨立的部件。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-08-new-8.16-joint-check.svg")))

下面是五份人工寫好的反例與停止紀錄。「內容」這欄只核對兩句需要抄的中文，額外英文另記在範圍；「正常EOS」是這個紙上紀錄給定的條件，不是模型實測成績。

| 手寫回答／停止條件 | 內容 | 範圍 | 格式 | 正常EOS |
| --- | --- | --- | --- | --- |
| 正確JSON，隨後選EOS | 通過 | 通過 | 通過 | 通過 |
| 把第一句寫成「今日開館」，其他與正確JSON相同，隨後選EOS | 未通過 | 通過 | 通過 | 通過 |
| JSON的清單照抄全部四行，隨後選EOS | 通過 | 未通過 | 通過 | 通過 |
| 只給兩行正確中文普通文字，隨後選EOS | 通過 | 通過 | 未通過 | 通過 |
| 可見文字是正確JSON，因生成上限而停，沒有EOS | 通過 | 通過 | 通過 | 未通過 |

第二份的抄寫範圍仍是兩句中文，但第一句有錯字；第三份沒有漏掉應抄的兩句，卻多抄英文。把它們分項記錄，就能知道是內容錯了，還是選取範圍不符，而不必把兩者都叫作「格式錯」。真實回答也可能一次違反兩項以上；本表故意一次改一件事，讓差別容易看見。

替這份任務準備示範時，同樣要把四項都寫進完整目標：兩句原文、限定範圍、指定JSON，以及答完之後的EOS。修正錯字不能替多抄英文解圍，補上JSON也不能替被截掉的EOS補回監督。先找出哪一項不符，再回到上一節的請求與目標配對逐筆修正。

整題成功不是四欄取平均。「三項通過」仍有一項必要要求沒有完成。對這份有限任務，只有第一份四項全過，才能算完成。若一次檢查多道題，應把每項通過題數和全部通過題數各自記錄，分母都包含那一批原定題目；不能先丟掉被上限截停的回答，再替剩下的答案算成功率。

規則也有它的邊界。能被JSON讀取，不等於只有指定欄位；搜尋到「今日休館」，不等於沒多抄英文；結尾出現某句文字，也不能證明模型正常選EOS。因此要先寫清楚接收端需要什麼，再選相應檢查。這個抄寫任務可以精確比對；換成開放式建議，多種說法都可能合理，不能把本節的字串規則直接當成通用內容裁判。

練習把要求改成「只抄英文，其餘格式相同」。保留原先那份中文JSON時，哪些項目仍通過？格式和正常結束可以保持通過，指定範圍與所需內容已不符。最後寫出英文目標，再核對四項，確認不是只改題目、卻拿舊答案當新題的成功示範。

<details>
<summary>補充：聯合驗收與IFEval的檢查器界線</summary>

[IFEval原論文](https://arxiv.org/pdf/2311.07911v1)§3區分逐項通過與同題全部要求通過；Table 1、§2.2說明可檢查要求與strict／loose評分。其[官方實作](https://github.com/google-research/google-research/blob/e49bbfe381c9c0e564b937f1c4e163a2273c65cc/instruction_following_eval/instructions.py)的JSON規則允許移除指定的Markdown外框，其他規則也有各自的判準。它們不等於本節「裸JSON、只有text欄位、值為字串清單」的全部要求，更不檢查圖片辨識或模型自然EOS。

正常結束另依[Transformers v4.57.1的生成設定](https://github.com/huggingface/transformers/blob/v4.57.1/src/transformers/generation/configuration_utils.py)區分eos_token_id、forced_eos_token_id與長度上限。相關段落與適用界線見[指令遵循來源筆記](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-revision-20261005/sources/instruction-following.md)。

</details>
